# 01 — Environment Setup & System Checks
## TeluguVoiceBridge v2 — Constrained Hardware Plan
**Hardware:** RTX 4060 Laptop (7.6 GB VRAM) | 24 GB RAM | ~60 GB Storage

This notebook:
1. Checks GPU, RAM, disk availability
2. Creates the project directory structure
3. Installs all dependencies in the correct order
4. Verifies every import
5. Writes YAML config files for all phases
6. Sets up swap space (optional)
7. Prints a VRAM budget summary

---
## 1.1 — System Health Checks

In [15]:
import subprocess, shutil, os, platform, psutil

print("="*60)
print("SYSTEM HEALTH CHECK")
print("="*60)

# --- OS & Python ---
print(f"OS:           {platform.system()} {platform.release()}")
print(f"Python:       {platform.python_version()}")

# --- GPU ---
try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap",
         "--format=csv,noheader"], text=True
    ).strip()
    print(f"GPU:          {gpu_info}")
except FileNotFoundError:
    print("GPU:          *** nvidia-smi NOT FOUND — no GPU detected ***")

# --- RAM ---
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"RAM:          {ram_gb:.1f} GB")

# --- Disk ---
PROJECT_ROOT = os.path.dirname(os.getcwd())  # Speech2/
disk = shutil.disk_usage(PROJECT_ROOT)
print(f"Disk free:    {disk.free / (1024**3):.1f} GB (on {PROJECT_ROOT})")

# --- Swap ---
swap_gb = psutil.swap_memory().total / (1024**3)
print(f"Swap:         {swap_gb:.1f} GB")

print("="*60)
# Sanity gates
assert ram_gb >= 16, f"Need ≥16 GB RAM, got {ram_gb:.1f} GB"
assert disk.free / (1024**3) >= 20, f"Need ≥20 GB free disk, got {disk.free/(1024**3):.1f} GB"
print("✓ All system checks passed.")

SYSTEM HEALTH CHECK
OS:           Linux 6.18.13-200.fc43.x86_64
Python:       3.11.6
GPU:          NVIDIA GeForce RTX 4060 Laptop GPU, 8188 MiB, 580.119.02, 8.9
RAM:          23.2 GB
Disk free:    29.4 GB (on /home/nibiru/Documents/sem6project/Speech2)
Swap:         8.0 GB
✓ All system checks passed.


---
## 1.2 — Directory Structure
Creates the full project tree. Safe to re-run (uses `exist_ok=True`).

In [2]:
import os, pathlib

BASE = pathlib.Path(os.getcwd())  # pipeline_v2/
print(f"Project root: {BASE}")

dirs = [
    # Data
    "data/raw",
    "data/processed/asr_train",
    "data/processed/speaker_train",
    "data/processed/tts_train",
    "data/processed/emotion_train",
    "data/metadata",
    "data/embeddings",
    # Checkpoints
    "checkpoints/whisper_merged",
    "checkpoints/speaker_encoder",
    "checkpoints/indictrans2_finetuned",
    "checkpoints/xtts_phase5a",
    "checkpoints/xtts_phase5b",
    "checkpoints/emotion_detector",
    # Misc
    "configs",
    "results/sample_outputs",
    "demo",
    "logs",
]

for d in dirs:
    (BASE / d).mkdir(parents=True, exist_ok=True)

print("Created directories:")
for d in sorted(dirs):
    print(f"  {d}/")
print(f"\n✓ {len(dirs)} directories ready.")

Project root: /home/nibiru/Documents/sem6project/Speech2/pipeline_v2
Created directories:
  checkpoints/emotion_detector/
  checkpoints/indictrans2_finetuned/
  checkpoints/speaker_encoder/
  checkpoints/whisper_merged/
  checkpoints/xtts_phase5a/
  checkpoints/xtts_phase5b/
  configs/
  data/embeddings/
  data/metadata/
  data/processed/asr_train/
  data/processed/emotion_train/
  data/processed/speaker_train/
  data/processed/tts_train/
  data/raw/
  demo/
  logs/
  results/sample_outputs/

✓ 17 directories ready.


---
## 1.3 — Install Dependencies

Run these cells **in order**. The install order matters to avoid dependency conflicts.

> **Note:** We use the existing `ml_env` conda environment with the already-installed PyTorch.
> If starting fresh, create `conda create -n ml_env python=3.10 -y` first.

In [17]:
# Step 1: Verify PyTorch + CUDA are already available
import torch, torchaudio
print(f"PyTorch:      {torch.__version__}")
print(f"Torchaudio:   {torchaudio.__version__}")
print(f"CUDA avail:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name:     {torch.cuda.get_device_name(0)}") 
assert torch.cuda.is_available(), "CUDA not available — check your driver/install"

PyTorch:      2.9.1+cu128
Torchaudio:   2.9.1+cu128
CUDA avail:   True
GPU name:     NVIDIA GeForce RTX 4060 Laptop GPU


In [18]:
# Step 2: Core ML libraries (transformers, peft, accelerate)
!pip install -q transformers peft accelerate

In [19]:
# Step 3: Quantization support
!pip install -q bitsandbytes

In [20]:
# Step 4: Dataset & audio processing
!pip install -q datasets soundfile librosa noisereduce pyloudnorm webrtcvad

In [21]:
# Step 5: SpeechBrain (speaker encoder) + Coqui TTS (XTTS-v2)
!pip install -q speechbrain TTS

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2025.11.6 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 4.5.0 which is incompatible.
unsloth-zoo 2025.11.6 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,<=4.57.2,>=4.51.3, but you have transformers 4.57.6 which is incompatible.
unsloth 2025.11.6 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 4.5.0 which is incompatible.
unsloth 2025.11.6 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.0,<=4.57.2,>=4.51.3, but you have transformers 4.57.6 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >=

In [22]:
# Step 6: Evaluation & utilities
!pip install -q sacrebleu jiwer gradio omegaconf audiomentations

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2025.11.6 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 4.5.0 which is incompatible.
unsloth-zoo 2025.11.6 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,<=4.57.2,>=4.51.3, but you have transformers 4.57.6 which is incompatible.
unsloth 2025.11.6 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 4.5.0 which is incompatible.
unsloth 2025.11.6 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.0,<=4.57.2,>=4.51.3, but you have transformers 4.57.6 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.5 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= 

In [23]:
# Step 7: UTMOS (TTS quality scoring)
# This installs from GitHub — may take a minute
!pip install -q git+https://github.com/sarulab-speech/UTMOS22.git 2>/dev/null || echo "UTMOS install skipped (optional)"

UTMOS install skipped (optional)


---
## 1.4 — Verify All Imports
A single cell that imports every library we need across all notebooks.

In [1]:
import importlib, sys

required = [
    ("torch",           "PyTorch"),
    ("torchaudio",      "Torchaudio"),
    ("transformers",    "HuggingFace Transformers"),
    ("peft",            "PEFT (LoRA)"),
    ("accelerate",      "Accelerate"),
    ("bitsandbytes",    "BitsAndBytes (quantization)"),
    ("datasets",        "HuggingFace Datasets"),
    ("soundfile",       "SoundFile"),
    ("librosa",         "Librosa"),
    ("noisereduce",     "NoiseReduce"),
    ("pyloudnorm",      "PyLoudNorm"),
    ("speechbrain",     "SpeechBrain"),
    ("TTS",             "Coqui TTS"),
    ("sacrebleu",       "SacreBLEU"),
    ("jiwer",           "JiWER (WER/CER)"),
    ("gradio",          "Gradio"),
    ("omegaconf",       "OmegaConf"),
    ("audiomentations", "Audiomentations"),
]

ok, fail = 0, 0
for mod, name in required:
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, "__version__", "?")
        print(f"  ✓ {name:<30s} {ver}")
        ok += 1
    except ImportError:
        print(f"  ✗ {name:<30s} ** MISSING **")
        fail += 1

print(f"\nResult: {ok}/{ok+fail} packages available.")
if fail:
    print("⚠ Re-run the install cells above for missing packages.")
else:
    print("✓ All imports verified.")

  ✓ PyTorch                        2.9.1+cu128
  ✓ Torchaudio                     2.9.1+cu128
  ✓ HuggingFace Transformers       4.57.6


2026-03-02 22:18:20.854777: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-02 22:18:20.880236: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-02 22:18:21.642380: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
Skipping import of cpp extensions due to incompatible torch version 2.9.1+cu128 for torchao

  ✓ PEFT (LoRA)                    0.18.1
  ✓ Accelerate                     1.12.0
  ✓ BitsAndBytes (quantization)    0.49.1
  ✓ HuggingFace Datasets           4.5.0
  ✓ SoundFile                      0.13.1
  ✓ Librosa                        0.11.0
  ✓ NoiseReduce                    ?
  ✓ PyLoudNorm                     ?
  ✓ SpeechBrain                    1.0.3
  ✓ Coqui TTS                      0.22.0
  ✓ SacreBLEU                      2.6.0
  ✓ JiWER (WER/CER)                ?
  ✓ Gradio                         6.8.0
  ✓ OmegaConf                      2.3.0
  ✓ Audiomentations                0.43.1

Result: 18/18 packages available.
✓ All imports verified.


---
## 1.5 — Write Configuration Files
Every training phase reads its config from YAML. No hardcoded values in training code.

In [3]:
import yaml, pathlib

CONFIGS_DIR = pathlib.Path("configs")

# ──────────────────────────────────────────────
# ASR (Whisper) config
# ──────────────────────────────────────────────
asr_cfg = {
    "model": {
        "name": "openai/whisper-large-v3",
        "load_in_8bit": True,
        "torch_dtype": "float16",
        "device_map": "cuda:0",
    },
    "lora": {
        "r": 8,
        "alpha": 16,
        "dropout": 0.05,
        "target_modules": ["q_proj", "v_proj"],
        "bias": "none",
    },
    "training": {
        "batch_size": 1,
        "gradient_accumulation_steps": 8,
        "max_steps": 5000,
        "warmup_steps": 200,
        "learning_rate": 1e-4,
        "lr_scheduler": "cosine",
        "fp16": True,
        "eval_steps": 250,
        "save_steps": 250,
        "save_total_limit": 2,
        "dataloader_num_workers": 2,
        "pin_memory": False,
        "language": "te",
        "task": "transcribe",
        "generation_max_length": 225,
    },
    "augmentation": {
        "spec_augment": {
            "time_masks": 2, "time_mask_max": 50,
            "freq_masks": 2, "freq_mask_max": 20,
            "apply_prob": 0.8,
        },
        "speed_perturbation": [0.9, 1.0, 1.1],
        "noise": {
            "snr_range": [15, 25],
            "apply_prob": 0.3,
        },
    },
    "target_metrics": {
        "wer": 0.20,
        "cer": 0.10,
    },
}

with open(CONFIGS_DIR / "asr.yaml", "w") as f:
    yaml.dump(asr_cfg, f, default_flow_style=False, sort_keys=False)
print("✓ configs/asr.yaml")

# ──────────────────────────────────────────────
# Speaker encoder config
# ──────────────────────────────────────────────
speaker_cfg = {
    "model": {
        "name": "speechbrain/spkrec-ecapa-voxceleb",
        "embedding_dim": 192,
        "device": "cpu",
    },
    "training": {
        "device": "cpu",
        "speakers_per_batch": 4,
        "utterances_per_speaker": 4,
        "learning_rate": 5e-5,
        "epochs": 20,
        "weight_decay": 1e-4,
        "early_stopping_patience": 5,
    },
    "loss": {
        "type": "aam_softmax",
        "margin": 0.2,
        "scale": 30,
    },
    "target_metrics": {
        "eer": 0.12,
        "cosine_same_speaker": 0.70,
        "inference_time_sec": 0.5,
    },
}

with open(CONFIGS_DIR / "speaker.yaml", "w") as f:
    yaml.dump(speaker_cfg, f, default_flow_style=False, sort_keys=False)
print("✓ configs/speaker.yaml")

# ──────────────────────────────────────────────
# Translation (IndicTrans2 / NLLB) config
# ──────────────────────────────────────────────
translation_cfg = {
    "model": {
        "name": "ai4bharat/indictrans2-indic-en-dist-200M",
        "fallback": "facebook/nllb-200-distilled-600M",
        "load_in_4bit": True,
        "torch_dtype": "float16",
        "device_map": "cuda:0",
        "src_lang": "tel_Telu",
        "tgt_lang": "eng_Latn",
    },
    "lora": {
        "r": 8,
        "alpha": 16,
        "target_modules": ["q_proj", "v_proj"],
    },
    "training": {
        "batch_size": 1,
        "gradient_accumulation_steps": 8,
        "epochs": 3,
        "learning_rate": 3e-5,
        "max_source_length": 256,
        "max_target_length": 256,
        "save_total_limit": 2,
    },
    "target_metrics": {
        "bleu_mt_only": 22,
        "bleu_pipeline": 15,
        "inference_time_sec": 2.0,
        "vram_gb": 1.5,
    },
}

with open(CONFIGS_DIR / "translation.yaml", "w") as f:
    yaml.dump(translation_cfg, f, default_flow_style=False, sort_keys=False)
print("✓ configs/translation.yaml")

# ──────────────────────────────────────────────
# Emotion detector config
# ──────────────────────────────────────────────
emotion_cfg = {
    "model": {
        "name": "facebook/wav2vec2-base",
        "freeze_layers": 6,
        "vad_output_dim": 3,
    },
    "training": {
        "batch_size": 4,
        "epochs": 20,
        "learning_rate": 1e-4,
        "weight_decay": 1e-4,
        "device": "cuda",
        "save_total_limit": 2,
    },
    "loss": {
        "type": "mse_ccc",
        "ccc_weight": 0.5,
    },
    "target_metrics": {
        "ccc": 0.55,
    },
}

with open(CONFIGS_DIR / "emotion.yaml", "w") as f:
    yaml.dump(emotion_cfg, f, default_flow_style=False, sort_keys=False)
print("✓ configs/emotion.yaml")

# ──────────────────────────────────────────────
# TTS (XTTS-v2) config
# ──────────────────────────────────────────────
tts_cfg = {
    "model": {
        "name": "tts_models/multilingual/multi-dataset/xtts_v2",
        "device": "cuda",
    },
    "phase5a": {
        "description": "Domain adaptation on LJSpeech + VCTK",
        "steps": 50000,
        "batch_size": 1,
        "learning_rate": 1e-4,
        "lr_decay_gamma": 0.999,
        "save_every": 5000,
        "save_total_limit": 2,
    },
    "phase5b": {
        "description": "Cross-lingual speaker generalization",
        "steps": 20000,
        "batch_size": 1,
        "learning_rate": 2e-5,
        "save_every": 2000,
        "save_total_limit": 2,
    },
    "target_metrics": {
        "utmos_5a": 3.5,
        "utmos_5b": 3.3,
        "secs": 0.70,
        "inference_time_sec": 5.0,
    },
}

with open(CONFIGS_DIR / "tts.yaml", "w") as f:
    yaml.dump(tts_cfg, f, default_flow_style=False, sort_keys=False)
print("✓ configs/tts.yaml")

print("\n✓ All 5 config files written to configs/")

✓ configs/asr.yaml
✓ configs/speaker.yaml
✓ configs/translation.yaml
✓ configs/emotion.yaml
✓ configs/tts.yaml

✓ All 5 config files written to configs/


---
## 1.6 — VRAM Budget Summary
Printable reference of GPU memory usage per phase.

In [ ]:
budget = """
╔══════════════════════════════════════════════════════════════════╗
║               VRAM BUDGET — 8 GB GPU (7.6 GB usable)          ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                ║
║  TRAINING (one model at a time):                               ║
║  ┌────────────────────────────────────────────────────┐        ║
║  │ Whisper-large-v3 INT8          2.5 GB              │        ║
║  │ LoRA trainable params          0.1 GB              │        ║
║  │ Optimizer states (AdamW)       0.4 GB              │        ║
║  │ Activations (batch=1)          1.5 GB              │        ║
║  │ CUDA overhead                  1.0 GB              │        ║
║  │ TOTAL                          5.5 GB  ✓           │        ║
║  └────────────────────────────────────────────────────┘        ║
║  ┌────────────────────────────────────────────────────┐        ║
║  │ Speaker Encoder (CPU only)     0.0 GB GPU          │        ║
║  │ ~80 MB RAM on CPU              ✓ runs in parallel  │        ║
║  └────────────────────────────────────────────────────┘        ║
║  ┌────────────────────────────────────────────────────┐        ║
║  │ IndicTrans2-200M 4-bit QLoRA   ~1.0 GB             │        ║
║  │ Optimizer + grads              ~2.0 GB             │        ║
║  │ CUDA overhead                  ~1.0 GB             │        ║
║  │ TOTAL                          ~4.0 GB  ✓          │        ║
║  └────────────────────────────────────────────────────┘        ║
║  ┌────────────────────────────────────────────────────┐        ║
║  │ XTTS-v2 model                  0.8 GB              │        ║
║  │ Activations (batch=1)          1.5 GB              │        ║
║  │ Optimizer states               2.0 GB              │        ║
║  │ CUDA overhead                  1.0 GB              │        ║
║  │ TOTAL                          5.3 GB  ✓           │        ║
║  └────────────────────────────────────────────────────┘        ║
║                                                                ║
║  INFERENCE (all models loaded):                                ║
║  ┌────────────────────────────────────────────────────┐        ║
║  │ Whisper INT8                   2.5 GB              │        ║
║  │ ECAPA-TDNN (CPU)               0.0 GB GPU          │        ║
║  │ IndicTrans2 4-bit              0.8 GB              │        ║
║  │ XTTS-v2                        0.8 GB              │        ║
║  │ CUDA overhead                  1.0 GB              │        ║
║  │ TOTAL                          5.1 GB ✓            │        ║
║  └────────────────────────────────────────────────────┘        ║
║                                                                ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(budget)


╔══════════════════════════════════════════════════════════════════╗
║               VRAM BUDGET — 8 GB GPU (7.6 GB usable)          ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                ║
║  TRAINING (one model at a time):                               ║
║  ┌────────────────────────────────────────────────────┐        ║
║  │ Whisper-large-v3 INT8          2.5 GB              │        ║
║  │ LoRA trainable params          0.1 GB              │        ║
║  │ Optimizer states (AdamW)       0.4 GB              │        ║
║  │ Activations (batch=1)          1.5 GB              │        ║
║  │ CUDA overhead                  1.0 GB              │        ║
║  │ TOTAL                          5.5 GB  ✓           │        ║
║  └────────────────────────────────────────────────────┘        ║
║  ┌────────────────────────────────────────────────────┐        ║
║  │ Speaker Encoder (CPU only)     0.0 GB GPU          │ 

---
## 1.7 — Swap Setup (Optional)
Adds 16 GB swap as a safety net. Requires sudo. Skip if already configured.

In [ ]:
import psutil

swap_gb = psutil.swap_memory().total / (1024**3)
print(f"Current swap: {swap_gb:.1f} GB")

if swap_gb < 8:
    print("\n⚠ Swap is low. To add 16 GB swap, run these commands in a terminal:")
    print("  sudo fallocate -l 16G /swapfile")
    print("  sudo chmod 600 /swapfile")
    print("  sudo mkswap /swapfile")
    print("  sudo swapon /swapfile")
    print("  # To make permanent: add '/swapfile none swap sw 0 0' to /etc/fstab")
else:
    print("✓ Swap is sufficient.")

---
## 1.8 — GPU Smoke Test
Quick test to confirm GPU training works with mixed-precision.

In [5]:
import torch
import torch.nn as nn

device = torch.device("cuda")

# Simple model → GPU → FP16 forward + backward
model = nn.Linear(256, 10).to(device)
x = torch.randn(4, 256, device=device)

with torch.amp.autocast("cuda"):
    out = model(x)
    loss = out.sum()

loss.backward()
print(f"Forward pass dtype: {out.dtype}")
print(f"Loss: {loss.item():.4f}")

# Cleanup
del model, x, out, loss
torch.cuda.empty_cache()

vram_used = torch.cuda.memory_allocated() / 1e6
print(f"VRAM after cleanup: {vram_used:.1f} MB")
print("✓ GPU smoke test passed.")

Forward pass dtype: torch.float16
Loss: -3.6832
VRAM after cleanup: 18.1 MB
✓ GPU smoke test passed.


---
## 1.9 — Storage Budget Reference
How our ~60 GB is allocated across the project.

In [ ]:
storage = """
╔══════════════════════════════════════════════════════════════╗
║              STORAGE BUDGET — 60 GB Total                  ║
╠══════════════════════════════════════════════════════════════╣
║                                                            ║
║  OS + Python env + libraries          ~8 GB                ║
║                                                            ║
║  DATASETS (processed, raw deleted):                        ║
║    CoVoST-2 Telugu (ASR + trans.)     ~2.5 GB              ║
║    Rasa Telugu (speaker + ASR)        ~1.5 GB              ║
║    VCTK (TTS multi-speaker)           ~8.0 GB              ║
║    LJSpeech (TTS baseline)            ~2.5 GB              ║
║    Subtotal datasets:                ~14.5 GB              ║
║                                                            ║
║  MODELS (quantized):                                       ║
║    Whisper-large-v3 INT8 + LoRA       ~1.6 GB              ║
║    ECAPA-TDNN speaker encoder         ~0.1 GB              ║
║    IndicTrans2-200M 4-bit             ~0.8 GB              ║
║    XTTS-v2 checkpoints (×2)           ~3.6 GB              ║
║    Emotion detector                   ~0.4 GB              ║
║    Subtotal models:                   ~6.5 GB              ║
║                                                            ║
║  WORKING SPACE:                                            ║
║    Processed audio (temp)            ~15.0 GB              ║
║    Training logs                      ~1.0 GB              ║
║    Embeddings                         ~0.1 GB              ║
║    Outputs + demo                     ~1.0 GB              ║
║    Subtotal working:                 ~17.1 GB              ║
║                                                            ║
║  BUFFER (always keep free):          ~13.9 GB              ║
║                                                            ║
║  TOTAL:                              ~60.0 GB              ║
╚══════════════════════════════════════════════════════════════╝

CRITICAL RULE: Download one dataset → process → delete raw → next.
Never have more than one raw dataset on disk at a time.
"""
print(storage)

---
## ✓ Notebook 01 Complete

**What we accomplished:**
- System checks passed (GPU, RAM, disk)
- Directory structure created
- All dependencies installed
- All imports verified
- 5 YAML config files written
- GPU smoke test passed
- VRAM & storage budgets documented

**Next:** Open `02_data_pipeline.ipynb` to download & preprocess datasets.